# Outliers (tempo × valor) — leitura do inteiro teor para modus operandi

**Pedido (Gilson, 08/07/2026):** segunda etapa da pesquisa — examinar o **inteiro teor** dos processos que são *outliers* por **tempo** e **valor**, para identificar padrões de atuação jurisdicional (*modus operandi*).

## Escopo deste notebook (fase exploratória)

1. Partir da amostra **FESP × ICMS** (round2) já filtrada no artigo EPED 2026.
2. Marcar outliers estatísticos de **valor corrigido (IPCA)** e **tempo até sentença** (IQR).
3. Priorizar casos **duplos** (alto valor **e** longa duração) para leitura qualitativa.
4. Recuperar texto disponível hoje no lake: `processos_delta.decisao` (decisões publicadas no CJPG).
5. Exportar fila de revisão manual em `output/`.

> **Performance:** todo o pipeline usa **lazy evaluation** (`scan_csv` / `scan_delta`). Só materializamos agregados pequenos e a fila final (`MAX_QUEUE_ROWS`). O texto `decisao` é buscado **apenas** para os processos da fila — nunca para o universo inteiro de outliers.
>
> **Limite atual dos dados:** o lake não tem PDF dos autos integrais; `decisao` cobre despachos/sentenças publicados. Coleta de inteiro teor integral = trabalho futuro.

## Setup

In [5]:
from __future__ import annotations

import json

import polars as pl

from config.paths import REPO_ROOT, SILVER_PROCESSOS

PLAYGROUND_ROOT = REPO_ROOT / "projects/litigancia/notebooks/playground"
OUTPUT_DIR = PLAYGROUND_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ARTICLE_CSV = (
    REPO_ROOT
    / "projects/litigancia/notebooks/articles/eped-2026-analise-execucoes-fiscais/artifacts"
    / "fesp_execucao_fiscal_icms_assuntos_filtered.csv"
)

QUEUE_CSV = OUTPUT_DIR / "outliers_tempo_valor_queue.csv"
STATS_JSON = OUTPUT_DIR / "outliers_tempo_valor_stats.json"

COHORT_COLUMNS = [
    "cd_processo",
    "numero",
    "assunto",
    "autores",
    "reus",
    "foro",
    "vara",
    "distribuicao_data",
    "data_sentenca_clean",
    "valor_corrigido_atual",
    "tempo_ate_sentenca_meses",
    "tipo_sentença",
]

IQR_MULTIPLIER = 1.5
MAX_QUEUE_ROWS = 200
PREVIEW_CHARS = 400


def tipo_ato_expr(col: str = "decisao") -> pl.Expr:
    """Primeira linha/trecho em maiúsculas de `decisao` (coluna não existe no delta)."""
    return (
        pl.col(col)
        .str.replace_all(r"\s+", " ")
        .str.extract(r"^([A-ZÁÉÍÓÚÃÕÇ\s]+)", 1)
        .str.strip_chars()
    )


## 1. Carregar amostra FESP-ICMS (lazy)

`scan_csv` + projeção de colunas leves — sem materializar o CSV inteiro na RAM.

In [6]:
lf_cohort = pl.scan_csv(
    ARTICLE_CSV,
    infer_schema_length=10_000,
    schema_overrides={"num_processo_limpo": pl.Utf8},
).select(COHORT_COLUMNS)

schema_cols = set(lf_cohort.collect_schema().names())
missing = [c for c in COHORT_COLUMNS if c not in schema_cols]
if missing:
    raise ValueError(f"Colunas ausentes no CSV: {missing}")

n_total = lf_cohort.select(pl.len()).collect(engine="streaming").item()
print(f"Amostra (lazy): {n_total:,} processos | colunas: {len(COHORT_COLUMNS)}")

Amostra (lazy): 292,920 processos | colunas: 12


## 2. Delimitar outliers (IQR)

Mesma lógica do [`fesp_execucao_fiscal_analysis.ipynb`](../articles/eped-2026-analise-execucoes-fiscais/notebooks/fesp_execucao_fiscal_analysis.ipynb) para valor; tempo usa IQR sobre `tempo_ate_sentenca_meses` (> 0).

In [7]:
def lazy_iqr_upper(
    col: str, lf: pl.LazyFrame, *, multiplier: float = IQR_MULTIPLIER
) -> float:
    qs = (
        lf.select(
            pl.col(col).quantile(0.25).alias("q1"),
            pl.col(col).quantile(0.75).alias("q3"),
        )
        .collect(engine="streaming")
        .row(0, named=True)
    )
    return float(qs["q3"] + multiplier * (qs["q3"] - qs["q1"]))


lf_valor_base = lf_cohort.filter(
    pl.col("valor_corrigido_atual").is_not_null()
    & (pl.col("valor_corrigido_atual") > 0)
)
lf_tempo_base = lf_cohort.filter(
    pl.col("tempo_ate_sentenca_meses").is_not_null()
    & (pl.col("tempo_ate_sentenca_meses") > 0)
)

limite_valor = lazy_iqr_upper("valor_corrigido_atual", lf_valor_base)
limite_tempo = lazy_iqr_upper("tempo_ate_sentenca_meses", lf_tempo_base)

lf_flags = lf_cohort.with_columns(
    [
        (
            pl.col("valor_corrigido_atual").is_not_null()
            & (pl.col("valor_corrigido_atual") > 0)
            & (pl.col("valor_corrigido_atual") > limite_valor)
        ).alias("outlier_valor"),
        (
            pl.col("tempo_ate_sentenca_meses").is_not_null()
            & (pl.col("tempo_ate_sentenca_meses") > 0)
            & (pl.col("tempo_ate_sentenca_meses") > limite_tempo)
        ).alias("outlier_tempo"),
    ]
).with_columns(
    pl.when(pl.col("outlier_valor") & pl.col("outlier_tempo"))
    .then(pl.lit("ambos"))
    .when(pl.col("outlier_valor"))
    .then(pl.lit("valor"))
    .when(pl.col("outlier_tempo"))
    .then(pl.lit("tempo"))
    .otherwise(pl.lit(None))
    .alias("outlier_tipo")
)

lf_outliers = lf_flags.filter(pl.col("outlier_tipo").is_not_null())

stats_row = (
    lf_flags.select(
        pl.len().alias("n_total"),
        pl.col("outlier_valor").sum().alias("n_outlier_valor"),
        pl.col("outlier_tempo").sum().alias("n_outlier_tempo"),
        (pl.col("outlier_tipo") == "ambos").sum().alias("n_outlier_ambos"),
    )
    .collect(engine="streaming")
    .row(0, named=True)
)

n_outlier_qualquer = lf_outliers.select(pl.len()).collect(engine="streaming").item()

stats = {
    "n_total": int(stats_row["n_total"]),
    "limite_valor_iqr": limite_valor,
    "limite_tempo_iqr_meses": limite_tempo,
    "n_outlier_valor": int(stats_row["n_outlier_valor"]),
    "n_outlier_tempo": int(stats_row["n_outlier_tempo"]),
    "n_outlier_ambos": int(stats_row["n_outlier_ambos"]),
    "n_outlier_qualquer": int(n_outlier_qualquer),
}

print(json.dumps(stats, indent=2, ensure_ascii=False))
print(
    lf_outliers.group_by("outlier_tipo")
    .len()
    .sort("len", descending=True)
    .collect(engine="streaming")
)

{
  "n_total": 292920,
  "limite_valor_iqr": 190827.72,
  "limite_tempo_iqr_meses": 464.7996057818659,
  "n_outlier_valor": 39520,
  "n_outlier_tempo": 1358,
  "n_outlier_ambos": 26,
  "n_outlier_qualquer": 40852
}
shape: (3, 2)
┌──────────────┬───────┐
│ outlier_tipo ┆ len   │
│ ---          ┆ ---   │
│ str          ┆ u32   │
╞══════════════╪═══════╡
│ valor        ┆ 39494 │
│ tempo        ┆ 1332  │
│ ambos        ┆ 26    │
└──────────────┴───────┘


## 3. Priorizar fila e buscar `decisao` só para o top-N

1. Ranqueia outliers em lazy (`lf_outliers`).
2. Materializa **apenas** `MAX_QUEUE_ROWS` linhas de metadados (sem texto).
3. `semi_join` em `processos_delta` traz `decisao` só desses `cd_processo`.

In [8]:
lf_queue_meta = (
    lf_outliers.with_columns(
        [
            (
                pl.col("valor_corrigido_atual").fill_null(0)
                * pl.col("tempo_ate_sentenca_meses").fill_null(0)
            ).alias("score_valor_x_tempo"),
            pl.when(pl.col("outlier_tipo") == "ambos")
            .then(pl.lit(0))
            .when(pl.col("outlier_tipo") == "valor")
            .then(pl.lit(1))
            .otherwise(pl.lit(2))
            .alias("prioridade_tipo"),
        ]
    )
    .sort(["prioridade_tipo", "score_valor_x_tempo"], descending=[False, True])
    .head(MAX_QUEUE_ROWS)
)

queue_meta = lf_queue_meta.collect(engine="streaming")
top_cds = queue_meta.select("cd_processo").unique()

lf_decisoes = (
    pl.scan_delta(str(SILVER_PROCESSOS))
    .join(top_cds.lazy(), on="cd_processo", how="inner")
    .select(
        "cd_processo",
        "id_processo",
        "decisao",
        "magistrado",
        "data_disponibilizacao",
    )
    .with_columns(
        [
            pl.col("decisao").str.len_chars().alias("decisao_len"),
            tipo_ato_expr().alias("tipo_ato"),
        ]
    )
    .sort(["cd_processo", "decisao_len"], descending=[False, True])
    .unique(subset=["cd_processo"], keep="first", maintain_order=True)
)

decisoes = lf_decisoes.collect(engine="streaming")

print(f"Fila priorizada: {queue_meta.height:,} processos")
print(f"Decisões recuperadas: {decisoes.height:,} / {top_cds.height:,}")
if decisoes.height:
    print(
        decisoes.select(
            pl.col("decisao_len").mean().alias("media_chars"),
            pl.col("decisao_len").median().alias("mediana_chars"),
            pl.col("decisao_len").max().alias("max_chars"),
        )
    )

Fila priorizada: 200 processos
Decisões recuperadas: 200 / 200
shape: (1, 3)
┌─────────────┬───────────────┬───────────┐
│ media_chars ┆ mediana_chars ┆ max_chars │
│ ---         ┆ ---           ┆ ---       │
│ f64         ┆ f64           ┆ u32       │
╞═════════════╪═══════════════╪═══════════╡
│ 1973.355    ┆ 1446.0        ┆ 14603     │
└─────────────┴───────────────┴───────────┘


## 4. Montar fila de revisão manual

Prioridade: `ambos` → score composto (valor × tempo) → demais outliers.

In [9]:
queue_out = (
    queue_meta.join(decisoes, on="cd_processo", how="left", suffix="_dec")
    .with_columns(
        [
            pl.col("decisao").fill_null("").str.len_chars().alias("decisao_len"),
            pl.col("decisao")
            .fill_null("")
            .str.replace_all(r"\s+", " ")
            .str.slice(0, PREVIEW_CHARS)
            .alias("decisao_preview"),
        ]
    )
    .with_row_index("fila_pos")
)

export_cols = [
    "fila_pos",
    "cd_processo",
    "numero",
    "outlier_tipo",
    "valor_corrigido_atual",
    "tempo_ate_sentenca_meses",
    "score_valor_x_tempo",
    "assunto",
    "autores",
    "reus",
    "foro",
    "vara",
    "distribuicao_data",
    "data_sentenca_clean",
    "tipo_sentença",
    "tipo_ato",
    "magistrado",
    "data_disponibilizacao",
    "decisao_len",
    "decisao_preview",
    "decisao",
]

queue_out = queue_out.select([c for c in export_cols if c in queue_out.columns])
queue_out.write_csv(QUEUE_CSV)

stats["queue_exportada"] = queue_out.height
stats["queue_csv"] = str(QUEUE_CSV)
STATS_JSON.write_text(
    json.dumps(stats, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)

print(f"Fila exportada: {QUEUE_CSV} ({queue_out.height} linhas)")
print(f"Stats: {STATS_JSON}")

XLSX_PATH = OUTPUT_DIR / "outliers_tempo_valor_revisao.xlsx"
import subprocess
import sys

subprocess.run(
    [sys.executable, str(REPO_ROOT / "pipelines/litigancia/scripts/playground/export_outliers_review_xlsx.py")],
    check=True,
)
print(f"XLSX exportado: {XLSX_PATH}")

queue_out.select(
    "fila_pos",
    "numero",
    "outlier_tipo",
    "valor_corrigido_atual",
    "tempo_ate_sentenca_meses",
    "foro",
    "decisao_len",
    "decisao_preview",
).head(10)


Fila exportada: /Users/etorebraga/Code/habitual-tax-debtor-research/notebooks/playground/output/outliers_tempo_valor_queue.csv (200 linhas)
Stats: /Users/etorebraga/Code/habitual-tax-debtor-research/notebooks/playground/output/outliers_tempo_valor_stats.json


fila_pos,numero,outlier_tipo,valor_corrigido_atual,tempo_ate_sentenca_meses,foro,decisao_len,decisao_preview
u32,str,str,f64,f64,str,u32,str
0,"""2050237-22.1985.8.26.0554""","""ambos""",8.6136e8,475.854139,"""Foro de Santo André""",0,""""""
1,"""2052345-24.1985.8.26.0554""","""ambos""",7.0342e8,474.835742,"""Foro de Santo André""",1013,"""SENTENÇA Processo nº: 2052345-…"
2,"""2050235-86.1984.8.26.0554""","""ambos""",1.3027e8,491.95138,"""Foro de Santo André""",1431,"""CONCLUSÃO Em , faço conclusão …"
3,"""2050239-26.1984.8.26.0554""","""ambos""",1.2747e8,482.851511,"""Foro de Santo André""",1448,"""CONCLUSÃO Em , faço conclusão …"
4,"""2050116-96.1982.8.26.0554""","""ambos""",7.9692e7,511.202365,"""Foro de Santo André""",1272,"""CONCLUSÃO Em , faço conclusão …"
5,"""2050098-41.1983.8.26.0554""","""ambos""",5.9690e7,483.837057,"""Foro de Santo André""",2351,"""SENTENÇA Processo Físico nº: 2…"
6,"""2050099-26.1983.8.26.0554""","""ambos""",5.5795e7,483.837057,"""Foro de Santo André""",2351,"""SENTENÇA Processo Físico nº: 2…"
7,"""2050088-94.1983.8.26.0554""","""ambos""",4.7585e7,478.613666,"""Foro de Santo André""",920,"""SENTENÇA Processo Físico nº: 2…"
8,"""2050090-64.1983.8.26.0554""","""ambos""",4.4776e7,504.99343,"""Foro de Santo André""",1446,"""CONCLUSÃO Em , faço conclusão …"


## 5. Leitura qualitativa — helper

Use `cd_processo` ou CNJ (`numero`) da fila para inspecionar o texto integral disponível.

In [10]:
def show_decisao(*, cd_processo: str | None = None, numero: str | None = None) -> None:
    """Busca lazy no lake — não depende de manter `decisao` na RAM."""
    if cd_processo is None and numero is None:
        raise ValueError("Informe cd_processo ou numero")

    if cd_processo is None:
        hit = (
            pl.scan_csv(QUEUE_CSV, schema_overrides={"num_processo_limpo": pl.Utf8})
            .filter(pl.col("numero") == numero)
            .select("cd_processo")
            .limit(1)
            .collect(engine="streaming")
        )
        if hit.is_empty():
            raise LookupError("numero não encontrado na fila exportada")
        cd_processo = hit["cd_processo"][0]

    meta = (
        pl.scan_csv(QUEUE_CSV, schema_overrides={"num_processo_limpo": pl.Utf8})
        .filter(pl.col("cd_processo") == cd_processo)
        .select(
            "numero",
            "outlier_tipo",
            "valor_corrigido_atual",
            "tempo_ate_sentenca_meses",
            "foro",
            "vara",
            "tipo_ato",
        )
        .limit(1)
        .collect(engine="streaming")
    )
    if meta.is_empty():
        raise LookupError("cd_processo não está na fila exportada")

    texto = (
        pl.scan_delta(str(SILVER_PROCESSOS))
        .filter(pl.col("cd_processo") == cd_processo)
        .select("decisao", "magistrado", "data_disponibilizacao")
        .with_columns(
            [
                pl.col("decisao").str.len_chars().alias("decisao_len"),
                tipo_ato_expr().alias("tipo_ato"),
            ]
        )
        .sort("decisao_len", descending=True)
        .limit(1)
        .collect(engine="streaming")
    )

    r = meta.row(0, named=True)
    tipo = r.get("tipo_ato")
    if texto.height and not tipo:
        tipo = texto["tipo_ato"][0]
    header = (
        f"{r['numero']} | {r['outlier_tipo']} | "
        f"R$ {r['valor_corrigido_atual']:,.2f} | {r['tempo_ate_sentenca_meses']:.1f} meses\n"
        f"{r['foro']} / {r['vara']} | {tipo or 'tipo_ato n/d'}\n"
        f"{'=' * 80}"
    )
    print(header)
    if texto.is_empty():
        print("[sem decisao no lake]")
    else:
        print(texto["decisao"][0])


# Exemplo: primeiro da fila (lazy via CSV exportado)
if queue_out.height:
    show_decisao(cd_processo=queue_out["cd_processo"][0])

2050237-22.1985.8.26.0554 | ambos | R$ 861,361,099.63 | 475.9 meses
Foro de Santo André / 1ª Vara da Fazenda Pública | tipo_ato n/d



## 6. Próximos passos (modus operandi)

Sugestão de codificação qualitativa após leitura piloto:

| Dimensão | Exemplos de códigos |
|---|---|
| Postura do juízo | rigoroso default / abertura a embargos / extinção rápida |
| Litigiosidade | citação frustrada / penhora repetida / acordo tardio |
| Credor (FESP) | peticionamento padronizado / pedidos excepcionais |
| Resultado | extinção por pagamento / prescrição / nulidade CDA |

**Trabalho futuro:**
- Scraping de peças/autos integrais no e-SAJ (fora do lake atual).
- Cruzar sequência de `movimentacoes_delta` com marcos temporais (citação → penhora → sentença).
- LLM assistido (`pydantic-ai`) só após piloto manual definir taxonomia — ver `CONTEXT.md`.